In [1]:
import os
import time
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits.csv'

Mounted at /content/data


In [ ]:
def extract_classical_features(dataframe, target_size=(64, 64)):
    """
    Carrega, converte para escala de cinza, redimensiona e achata as imagens.
    """
    features = []
    labels = []
    
    # Barra de progresso tqdm
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc="Extraindo imagens"):
        img_path = row['image_path']
        label = row['label']
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
            
        img_resized = cv2.resize(img, target_size)
        flattened_features = img_resized.flatten()
        
        features.append(flattened_features)
        labels.append(label)
        
    return np.array(features), np.array(labels)

In [ ]:
def load_fold_from_csv(csv_path, target_fold_idx):
    """
    Lê o arquivo CSV e filtra as linhas do fold atual.
    """
    df = pd.read_csv(csv_path)
    fold_df = df[df['fold'] == target_fold_idx]
    train_df = fold_df[fold_df['split_type'] == 'train']
    val_df = fold_df[fold_df['split_type'] == 'val']
    return train_df, val_df

In [ ]:
def train_evaluate_xgboost(csv_path, label_encoder, num_folds=5, image_size=(64, 64)):
    print(f"Iniciando avaliação do XGBoost com imagens {image_size}...")
    fold_metrics = []
    
    # Parâmetros padrão robustos para o XGBoost
    xgb_classifier = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42, 
        n_jobs=-1
    )
    
    for fold_idx in range(num_folds):
        print(f"\n{'='*40}")
        print(f"Processando Fold {fold_idx + 1}/{num_folds} (XGBoost)")
        print(f"{'='*40}")
        
        # Lendo diretamente do CSV
        train_df, val_df = load_fold_from_csv(csv_path, target_fold_idx=fold_idx)
        
        print("Conjunto de Treinamento:")
        X_train, y_train_text = extract_classical_features(train_df, target_size=image_size)
        print("Conjunto de Validação:")
        X_val, y_val_text = extract_classical_features(val_df, target_size=image_size)
        
        # Conversão de texto para números
        y_train = label_encoder.transform(y_train_text)
        y_val = label_encoder.transform(y_val_text)
        
        print("Treinando modelo XGBoost...")
        start_time = time.time()
        xgb_classifier.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        print("Avaliando...")
        y_pred = xgb_classifier.predict(X_val)
        
        fold_accuracy = accuracy_score(y_val, y_pred)
        print(f"Fold {fold_idx + 1} Acurácia: {fold_accuracy:.4f} (Tempo de treino: {training_time:.2f}s)")
        
        # Mantendo labels=[0, 1, 2, 3] para evitar o erro da classe 'fantasia' vazia no Fold 5
        target_names = label_encoder.classes_
        print(classification_report(y_val, y_pred, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))
        
        fold_metrics.append(fold_accuracy)
        
    avg_accuracy = np.mean(fold_metrics)
    std_accuracy = np.std(fold_metrics)
    
    print(f"\n{'='*40}")
    print("RESULTADO FINAL CROSS-VALIDATION - XGBOOST")
    print(f"Acurácia Média: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"{'='*40}\n")
    
    return xgb_classifier, fold_metrics

In [6]:
if __name__ == "__main__":
    if os.path.exists(SPLITS_CSV_PATH):
        # A ordenação é importante para o XGBoost alinhar perfeitamente com 0, 1, 2, 3
        classes = sorted(['grotesco', 'serifado', 'escritural', 'fantasia'])
        label_encoder = LabelEncoder()
        label_encoder.fit(classes)
        
        trained_xgb, xgb_accuracies = train_evaluate_xgboost(
            csv_path=SPLITS_CSV_PATH, 
            label_encoder=label_encoder,
            num_folds=5,
            image_size=(64, 64)
        )
    else:
        print(f"Erro: O arquivo '{SPLITS_CSV_PATH}' não foi encontrado. Execute o script de divisão do passo 1 primeiro.")

Iniciando avaliação do XGBoost com imagens (64, 64)...

Processando Fold 1/5 (XGBoost)
Conjunto de Treinamento:


Extraindo imagens: 100%|██████████| 919/919 [00:04<00:00, 222.11it/s]


Conjunto de Validação:


Extraindo imagens: 100%|██████████| 212/212 [00:00<00:00, 212.51it/s]


Treinando modelo XGBoost...
Avaliando...
Fold 1 Acurácia: 0.5896 (Tempo de treino: 192.73s)
              precision    recall  f1-score   support

  escritural       0.83      0.66      0.73        94
    fantasia       0.00      0.00      0.00        11
    grotesco       0.40      0.76      0.52        54
    serifado       0.67      0.42      0.51        53

    accuracy                           0.59       212
   macro avg       0.47      0.46      0.44       212
weighted avg       0.63      0.59      0.59       212


Processando Fold 2/5 (XGBoost)
Conjunto de Treinamento:


...

Treinando modelo XGBoost...
Avaliando...
Fold 4 Acurácia: 0.7452 (Tempo de treino: 192.66s)
              precision    recall  f1-score   support

  escritural       0.61      0.82      0.70        57
    fantasia       0.00      0.00      0.00        13
    grotesco       0.86      0.93      0.89       150
    serifado       0.39      0.21      0.27        43

    accuracy                           0.75       263
   macro avg       0.47      0.49      0.47       263
weighted avg       0.69      0.75      0.71       263


Processando Fold 5/5 (XGBoost)
Conjunto de Treinamento:


Extraindo imagens: 100%|██████████| 892/892 [00:03<00:00, 239.72it/s]


Conjunto de Validação:


Extraindo imagens: 100%|██████████| 239/239 [00:00<00:00, 241.89it/s]


Treinando modelo XGBoost...
Avaliando...
Fold 5 Acurácia: 0.8368 (Tempo de treino: 204.64s)
              precision    recall  f1-score   support

  escritural       0.84      0.95      0.89        96
    fantasia       0.00      0.00      0.00         0
    grotesco       0.85      0.94      0.89       101
    serifado       0.74      0.33      0.46        42

    accuracy                           0.84       239
   macro avg       0.61      0.56      0.56       239
weighted avg       0.83      0.84      0.82       239


RESULTADO FINAL CROSS-VALIDATION - XGBOOST
Acurácia Média: 0.7449 ± 0.1025

